# Notebook 02 - Tiền xử lý văn bản

Notebook này tạo dữ liệu sạch cho các bước TF-IDF, K-Means và SVM. Kết quả chính là `datasets/cleaned/train_clean.csv` và `datasets/cleaned/test_clean.csv`.

## 0. Khởi tạo môi trường

In [1]:
from pathlib import Path
import os
import sys
import warnings

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
os.chdir(PROJECT_ROOT)
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

FIG_DIR = Path("results/figures")
CSV_DIR = Path("results/csv")
MODEL_DIR = Path("results/models")
for path in [FIG_DIR, CSV_DIR, MODEL_DIR, Path("datasets/cleaned")]:
    path.mkdir(parents=True, exist_ok=True)

warnings.filterwarnings("ignore")

## 2.1 Áp dụng pipeline tiền xử lý

Pipeline chạy đúng thứ tự: lowercase, remove_punctuation, normalize_numbers, remove_stopword, ViTokenizer, remove_repeated_words. Ví dụ dưới đây minh họa rõ tác dụng của từng bước.

In [2]:
import pandas as pd
from IPython.display import display

from src.preprocessing import (
    add_aspect_columns,
    add_sentiment_columns,
    preprocess_dataframe,
    preprocessing_steps_example,
    read_filestopwords,
)

train_raw = pd.read_csv("datasets/raw/Train.csv", encoding="utf-8")
test_raw = pd.read_csv("datasets/raw/Test.csv", encoding="utf-8")
stop_words = read_filestopwords("datasets/stopwords/vietnamese-stopwords.txt")

example = "Mua máy này 2 tháng pin tụt quá nhanh!!"
display(preprocessing_steps_example(example, stop_words))

,Step,Result
0,Before,Mua máy này 2 tháng pin tụt quá nhanh!!
1,Step 1: lowercase,mua máy này 2 tháng pin tụt quá nhanh!!
2,Step 2: remove punctuation,mua máy này 2 tháng pin tụt quá nhanh
3,Step 3: normalize numbers,mua máy này number tháng pin tụt quá nhanh
4,Step 4: remove stopwords,mua máy number tháng pin tụt quá nhanh
5,Step 5: tokenize,mua máy number tháng pin tụt quá nhanh
6,Step 6: remove repeated words,mua máy number tháng pin tụt quá nhanh


## 2.2 Gán nhãn tổng thể

Từ chuỗi aspect-based trong cột `label`, ta đếm `positive_count`, `negative_count`, `neutral_count` rồi gán nhãn tổng thể `sentiment`.

In [3]:
train_labeled = add_sentiment_columns(train_raw)
test_labeled = add_sentiment_columns(test_raw)

print("Phân phối sentiment sau khi gán nhãn tổng thể - Train:")
display(train_labeled["sentiment"].value_counts().to_frame("count"))
display((train_labeled["sentiment"].value_counts(normalize=True) * 100).round(2).to_frame("percentage"))
print("Nhận xét: Phân phối này quyết định cách đánh giá mô hình; nếu lệch lớp thì F1 Macro quan trọng hơn Accuracy.")

Phân phối sentiment sau khi gán nhãn tổng thể - Train:


,count
sentiment,
Positive,4008
Negative,2966
Neutral,812


,percentage
sentiment,
Positive,51.48
Negative,38.09
Neutral,10.43


Nhận xét: Phân phối này quyết định cách đánh giá mô hình; nếu lệch lớp thì F1 Macro quan trọng hơn Accuracy.


## 2.3 Trích xuất aspect chính

`main_aspect` lấy aspect đầu tiên có ý nghĩa khác `OTHERS`. `aspects_list` giữ toàn bộ cặp `(aspect, sentiment)` để phục vụ EDA và phân tích sau này.

In [4]:
train_aspect = add_aspect_columns(train_labeled)
display(train_aspect[["label", "sentiment", "main_aspect", "aspects_list"]].head())

aspect_dist = train_aspect["main_aspect"].value_counts()
display(aspect_dist)
print(f"Nhận xét: main_aspect phổ biến nhất là {aspect_dist.index[0]}, đây có thể là chủ đề lớn nhất khi so sánh với K-Means.")

,label,sentiment,main_aspect,aspects_list
0,{CAMERA#Positive};{FEATURES#Positive};{BATTERY...,Positive,CAMERA,"[(CAMERA, Positive), (FEATURES, Positive), (BA..."
1,{BATTERY#Negative};{GENERAL#Positive};{OTHERS};,Neutral,BATTERY,"[(BATTERY, Negative), (GENERAL, Positive)]"
2,{FEATURES#Negative};,Negative,FEATURES,"[(FEATURES, Negative)]"
3,{FEATURES#Negative};{BATTERY#Neutral};{GENERAL...,Neutral,FEATURES,"[(FEATURES, Negative), (BATTERY, Neutral), (GE..."
4,{BATTERY#Positive};{PERFORMANCE#Positive};{SER...,Positive,BATTERY,"[(BATTERY, Positive), (PERFORMANCE, Positive),..."


main_aspect
CAMERA         1750
FEATURES       1669
BATTERY        1455
PERFORMANCE     989
SCREEN          949
GENERAL         309
PRICE           262
DESIGN          195
OTHERS          115
SER&ACC          85
STORAGE           8
Name: count, dtype: int64

Nhận xét: main_aspect phổ biến nhất là CAMERA, đây có thể là chủ đề lớn nhất khi so sánh với K-Means.


## 2.4 Kiểm tra sau tiền xử lý

Chạy pipeline cho toàn bộ train/test, kiểm tra số dòng rỗng sau clean, phân phối độ dài và một vài ví dụ trước/sau.

In [5]:
train_clean_full = preprocess_dataframe(train_raw, stop_words)
test_clean_full = preprocess_dataframe(test_raw, stop_words)

empty_train = (train_clean_full["comment"].str.strip() == "").sum()
empty_test = (test_clean_full["comment"].str.strip() == "").sum()
print(f"Số dòng train rỗng sau clean: {empty_train}")
print(f"Số dòng test rỗng sau clean : {empty_test}")

train_clean_full = train_clean_full[train_clean_full["comment"].str.strip() != ""].copy()
test_clean_full = test_clean_full[test_clean_full["comment"].str.strip() != ""].copy()

display(train_clean_full["word_count"].describe().round(2))
display(train_clean_full[["raw_comment", "comment", "sentiment", "main_aspect"]].head(8))
print("Nhận xét: Sau tiền xử lý, câu ngắn gọn hơn, số được chuẩn hóa thành `number`, và các cụm tiếng Việt có thể được nối bằng dấu gạch dưới.")

Số dòng train rỗng sau clean: 0
Số dòng test rỗng sau clean : 0


count    7786.00
mean       24.85
std        13.56
min         1.00
25%        16.00
50%        21.00
75%        29.75
max       128.00
Name: word_count, dtype: float64

,raw_comment,comment,sentiment,main_aspect
0,Mới mua máy này Tại thegioididong thốt nốt cảm...,mới mua máy thegioididong thốt_nốt cảm_thấy ok...,Positive,CAMERA
1,Pin kém còn lại miễn chê mua 8/3/2019 tình trạ...,pin kém còn miễn chê mua number tình_trạng pin...,Neutral,BATTERY
2,Sao lúc gọi điện thoại màn hình bị chấm nhỏ nh...,sao gọi điện_thoại màn_hình chấm nhỏ nháy gần ...,Negative,FEATURES
3,"Mọi người cập nhật phần mềm lại , nó sẽ bớt tố...",mọi người cập_nhật phần_mềm nó bớt tốn pin mìn...,Neutral,FEATURES
4,"Mới mua Sài được 1 tháng thấy pin rất trâu, Sà...",mới mua sài number tháng thấy pin trâu sài bao...,Positive,BATTERY
5,"Xài tốt, rất mượt, pin trâu nếu các bạn để độ ...",xài tốt mượt pin trâu bạn độ sáng đủ nhân_viên...,Positive,BATTERY
6,Mình mới xài được 7 tháng xuống 7% pin. Chả hi...,mình mới xài number tháng xuống number pin chả...,Negative,BATTERY
7,Hôm qua ngày 23/6/2020 e ra thế giới di động ...,hôm ngày number e thế_giới di_động mua dthoai ...,Negative,SER&ACC


Nhận xét: Sau tiền xử lý, câu ngắn gọn hơn, số được chuẩn hóa thành `number`, và các cụm tiếng Việt có thể được nối bằng dấu gạch dưới.


## 2.5 Lưu dữ liệu sạch

In [6]:
keep_cols = ["comment", "sentiment", "main_aspect", "n_star", "aspects_list"]
train_clean_full[keep_cols].to_csv("datasets/cleaned/train_clean.csv", index=False, encoding="utf-8-sig")
test_clean_full[keep_cols].to_csv("datasets/cleaned/test_clean.csv", index=False, encoding="utf-8-sig")

print("Đã lưu datasets/cleaned/train_clean.csv:", train_clean_full[keep_cols].shape)
print("Đã lưu datasets/cleaned/test_clean.csv :", test_clean_full[keep_cols].shape)

Đã lưu datasets/cleaned/train_clean.csv: (7786, 5)


Đã lưu datasets/cleaned/test_clean.csv : (2224, 5)
